# Amini Soil Prediction

Predict eleven soil nutrient concentrations, then convert those predictions into nutrient gaps for the competition submission. Run the cells in order after placing `Train.csv`, `Test.csv`, and `Gap_Test.csv` in `data/`, or set `AMINI_DATA_DIR` to their directory.

**Workflow:** load and validate inputs → engineer spectral indices → inspect features → fit and compare models on one holdout split → refit the chosen model on all labeled data → write `submission.csv`.

This notebook contains model implementations, not verified competition results. The original experiments and their scores cannot be recovered from the surviving notebook. `VariableDefinitions.csv` is optional for the exploratory section.

## 1. Environment and configuration

Install dependencies in the current kernel if necessary: `%pip install numpy pandas matplotlib seaborn scikit-learn xgboost lightgbm`. Restart the kernel after installation. XGBoost and LightGBM are imported only when their model cells run; the three regularized linear models and Random Forest need scikit-learn.

Use a local `data/` folder by default. For Google Colab, mount Drive yourself and set `AMINI_DATA_DIR` before running this notebook; no personal Drive path or authentication is stored here.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import ElasticNetCV, LassoCV, RidgeCV
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.multioutput import MultiOutputRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
DATA_DIR = Path(os.environ.get("AMINI_DATA_DIR", "data")).expanduser()
OUTPUT_PATH = Path("submission.csv")
TARGET_COLUMNS = ["N", "P", "K", "Ca", "Mg", "S", "Fe", "Mn", "Zn", "Cu", "B"]
ID_COLUMN = "PID"
SOIL_DEPTH_CM = 20  # Original notebook's assumed sampling depth.

## 2. Load and validate data

Keep target columns out of the predictors. Drop rows without any complete set of labels, so imputation never invents ground-truth nutrient values. Feature imputation happens *inside* each model pipeline, using only the training fold during validation.

In [ ]:
def read_required_csv(filename):
    path = DATA_DIR / filename
    if not path.is_file():
        raise FileNotFoundError(f"Missing {path}. Set AMINI_DATA_DIR or supply the CSV files.")
    return pd.read_csv(path)


train_df = read_required_csv("Train.csv")
test_df = read_required_csv("Test.csv")
gap_test_df = read_required_csv("Gap_Test.csv")

for name, frame, required in (
    ("Train.csv", train_df, [ID_COLUMN, *TARGET_COLUMNS, "mb1", "mb2", "mb3"]),
    ("Test.csv", test_df, [ID_COLUMN, "BulkDensity", "mb1", "mb2", "mb3"]),
    ("Gap_Test.csv", gap_test_df, [ID_COLUMN, "Nutrient", "Required"]),
):
    missing = sorted(set(required) - set(frame.columns))
    if missing:
        raise ValueError(f"{name} is missing required columns: {missing}")

if train_df[ID_COLUMN].duplicated().any() or test_df[ID_COLUMN].duplicated().any():
    raise ValueError("PID must uniquely identify each train/test sample.")
if gap_test_df[[ID_COLUMN, "Nutrient"]].duplicated().any():
    raise ValueError("Gap_Test.csv has duplicate PID/Nutrient pairs.")

train_df = train_df.dropna(subset=TARGET_COLUMNS).copy()
print(f"Labeled rows: {len(train_df):,}; test rows: {len(test_df):,}; gaps: {len(gap_test_df):,}")

## 3. Feature engineering

Retain the original notebook's NDVI and EVI formulas and its removal of source bands `mb1`, `mb2`, `mb3`, `mb7`, and `parv`. Undefined index values from zero denominators become missing values and are imputed during fitting. Exclude `site` and `PID` from learning; `PID` is kept separately for submission joins.

In [ ]:
def engineer_features(frame):
    result = frame.copy()
    mb1 = pd.to_numeric(result["mb1"], errors="coerce")
    mb2 = pd.to_numeric(result["mb2"], errors="coerce")
    mb3 = pd.to_numeric(result["mb3"], errors="coerce")
    with np.errstate(divide="ignore", invalid="ignore"):
        result["ndvi"] = (mb2 - mb1) / (mb2 + mb1)
        result["evi"] = 2.5 * (mb2 - mb1) / (mb2 + 6 * mb1 - 7.5 * mb3 + 1)
    result = result.drop(columns=["mb1", "mb2", "mb3", "mb7", "parv"], errors="ignore")
    return result.replace([np.inf, -np.inf], np.nan)


train_features = engineer_features(train_df)
test_features = engineer_features(test_df)
feature_columns = [
    col for col in train_features.columns if col not in [*TARGET_COLUMNS, ID_COLUMN, "site"]
]
missing_features = sorted(set(feature_columns) - set(test_features.columns))
if missing_features:
    raise ValueError(f"Test.csv lacks training features: {missing_features}")

X = train_features[feature_columns].copy()
X_test = test_features[feature_columns].copy()
y = train_df[TARGET_COLUMNS].apply(pd.to_numeric, errors="coerce")
if y.isna().any().any() or not np.isfinite(y.to_numpy()).all():
    raise ValueError("All target values must be finite and numeric after loading.")
if len(X) < 7:
    raise ValueError("At least seven labeled rows are needed for the holdout and five-fold CV.")
print(f"Features: {X.shape[1]}; targets: {len(TARGET_COLUMNS)}")

## 4. Exploratory checks

These plots describe the labeled set and spatial coverage. They do not select features or fit transforms. The variance plot covers numeric predictors only; variance magnitude depends on measurement units.

In [ ]:
numeric_preview = X.select_dtypes(include=np.number)
if not numeric_preview.empty:
    numeric_preview.var().sort_values(ascending=False).plot.bar(figsize=(12, 4))
    plt.title("Numeric feature variance (unscaled)")
    plt.ylabel("Variance")
    plt.tight_layout()
    plt.show()

correlations = train_features.drop(columns=[ID_COLUMN, "site"], errors="ignore").corr(
    numeric_only=True, method="spearman"
)
plt.figure(figsize=(11, max(6, 0.35 * len(correlations))))
sns.heatmap(correlations[TARGET_COLUMNS], cmap="coolwarm", center=0)
plt.title("Spearman correlation with soil nutrients")
plt.tight_layout()
plt.show()

if {"lon", "lat"}.issubset(train_df.columns) and {"lon", "lat"}.issubset(test_df.columns):
    plt.figure(figsize=(8, 6))
    plt.scatter(train_df["lon"], train_df["lat"], label="Train", alpha=0.3)
    plt.scatter(test_df["lon"], test_df["lat"], label="Test", alpha=0.5)
    plt.xlabel("Longitude")
    plt.ylabel("Latitude")
    plt.legend()
    plt.tight_layout()
    plt.show()

definitions_path = DATA_DIR / "VariableDefinitions.csv"
if definitions_path.is_file():
    display(pd.read_csv(definitions_path).head())

## 5. Holdout split and preprocessing

All candidates use the same random 80/20 holdout (`random_state=42`). This gives a comparable offline score; nearby samples or shared sites can make a random split optimistic for new locations. `RidgeCV`, `LassoCV`, and `ElasticNetCV` select regularization strength using five folds *within the training partition*. Each nutrient gets its own tuned estimator. Numeric inputs are imputed and standardized; categorical inputs are imputed and one-hot encoded.

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)
CV_FOLDS = min(5, len(X_train))

numeric_columns = X.select_dtypes(include=np.number).columns.tolist()
categorical_columns = X.columns.difference(numeric_columns).tolist()
if not numeric_columns and not categorical_columns:
    raise ValueError("No predictor columns were found.")


def make_preprocessor(scale_numeric=True):
    transformers = []
    if numeric_columns:
        numeric_steps = [("imputer", SimpleImputer(strategy="median", keep_empty_features=True))]
        if scale_numeric:
            numeric_steps.append(("scaler", StandardScaler()))
        transformers.append(("numeric", Pipeline(numeric_steps), numeric_columns))
    if categorical_columns:
        categorical_steps = Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent", keep_empty_features=True)),
            ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ])
        transformers.append(("categorical", categorical_steps, categorical_columns))
    return ColumnTransformer(transformers, sparse_threshold=0)


def make_pipeline(estimator, *, scale_numeric=True):
    return Pipeline([
        ("preprocess", make_preprocessor(scale_numeric=scale_numeric)),
        ("regressor", estimator),
    ])

## 6. Candidate models and regularization

- **RidgeCV:** L2 shrinkage, with one selected `alpha` per nutrient.
- **LassoCV:** L1 shrinkage and sparse coefficients.
- **ElasticNetCV:** combined L1/L2 regularization, tuning `alpha` and `l1_ratio`.
- **Random Forest:** use `min_samples_leaf` and `max_features` to limit overfitting.
- **XGBoost:** L1 (`reg_alpha`) and L2 (`reg_lambda`) penalties plus depth and subsampling controls.
- **LightGBM:** L1 (`reg_alpha`) and L2 (`reg_lambda`) penalties plus leaf, depth, and sampling controls.

Set `RUN_OPTIONAL_BOOSTERS = True` after installing XGBoost and LightGBM. Candidate factories create fresh estimators for validation and the later full-data refit. `CV_FOLDS` must be at least 2.

In [ ]:
if CV_FOLDS < 2:
    raise ValueError("The training partition needs at least two rows for cross-validation.")


def ridge_model():
    return make_pipeline(MultiOutputRegressor(
        RidgeCV(alphas=np.logspace(-3, 4, 30), cv=CV_FOLDS)
    ))


def lasso_model():
    return make_pipeline(MultiOutputRegressor(
        LassoCV(cv=CV_FOLDS, max_iter=20000, random_state=RANDOM_STATE)
    ))


def elastic_net_model():
    return make_pipeline(MultiOutputRegressor(
        ElasticNetCV(l1_ratio=[0.1, 0.25, 0.5, 0.75, 0.9, 0.99],
                     cv=CV_FOLDS, max_iter=20000, random_state=RANDOM_STATE)
    ))


def random_forest_model():
    return make_pipeline(
        RandomForestRegressor(n_estimators=450, min_samples_leaf=2,
                              max_features=0.8, n_jobs=-1, random_state=RANDOM_STATE),
        scale_numeric=False,
    )


def xgboost_model():
    from xgboost import XGBRegressor

    estimator = XGBRegressor(
        n_estimators=1000, learning_rate=0.01, max_depth=5,
        subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=1.0,
        tree_method="hist", n_jobs=-1, random_state=RANDOM_STATE,
    )
    return make_pipeline(MultiOutputRegressor(estimator), scale_numeric=False)


def lightgbm_model():
    from lightgbm import LGBMRegressor

    estimator = LGBMRegressor(
        n_estimators=400, learning_rate=0.01, num_leaves=31,
        max_depth=8, min_child_samples=20, subsample=0.8,
        subsample_freq=1, colsample_bytree=0.8,
        reg_alpha=0.1, reg_lambda=1.0, n_jobs=-1,
        random_state=RANDOM_STATE, verbosity=-1,
    )
    return make_pipeline(MultiOutputRegressor(estimator), scale_numeric=False)


RUN_OPTIONAL_BOOSTERS = False
MODEL_FACTORIES = {
    "RidgeCV": ridge_model,
    "LassoCV": lasso_model,
    "ElasticNetCV": elastic_net_model,
    "RandomForest": random_forest_model,
}
if RUN_OPTIONAL_BOOSTERS:
    MODEL_FACTORIES.update({"XGBoost": xgboost_model, "LightGBM": lightgbm_model})

## 7. Fit, validate, and inspect

Report per-nutrient RMSE and MAE and the unweighted mean across nutrients. The mean is a convenient local comparison, **not a claim about Zindi's official metric**. Targets have different units and scales; choose `SELECTED_MODEL` explicitly if the competition metric or domain priorities differ. The holdout is used once for comparing candidates, so its score is a model-selection estimate rather than a separate final test score.

In [ ]:
validation_models = {}
validation_predictions = {}
metric_rows = []
for name, factory in MODEL_FACTORIES.items():
    fitted = factory().fit(X_train, y_train)
    predictions = np.asarray(fitted.predict(X_val))
    if predictions.shape != y_val.shape or not np.isfinite(predictions).all():
        raise ValueError(f"{name} returned invalid validation predictions: {predictions.shape}")
    validation_models[name] = fitted
    validation_predictions[name] = predictions
    for column_index, nutrient in enumerate(TARGET_COLUMNS):
        actual = y_val[nutrient].to_numpy()
        predicted = predictions[:, column_index]
        metric_rows.append({
            "model": name,
            "nutrient": nutrient,
            "rmse": np.sqrt(mean_squared_error(actual, predicted)),
            "mae": mean_absolute_error(actual, predicted),
        })

metrics = pd.DataFrame(metric_rows)
summary = metrics.groupby("model")[["rmse", "mae"]].mean().sort_values("rmse")
display(summary)
display(metrics.sort_values(["model", "nutrient"]))

In [ ]:
# Inspect per-target regularization parameters after fitting the CV estimators.
for name in ("RidgeCV", "LassoCV", "ElasticNetCV"):
    regressors = validation_models[name].named_steps["regressor"].estimators_
    tuned = pd.DataFrame({
        "nutrient": TARGET_COLUMNS,
        "alpha": [est.alpha_ for est in regressors],
    })
    if name == "ElasticNetCV":
        tuned["l1_ratio"] = [est.l1_ratio_ for est in regressors]
    print(f"{name} chosen penalties")
    display(tuned)

SELECTED_MODEL = summary.index[0]  # Override with a key from MODEL_FACTORIES if appropriate.
nitrogen_index = TARGET_COLUMNS.index("N")
nitrogen_residuals = (
    y_val["N"].to_numpy() - validation_predictions[SELECTED_MODEL][:, nitrogen_index]
)
plt.figure(figsize=(7, 5))
sns.scatterplot(x=y_val["N"].to_numpy(), y=nitrogen_residuals, alpha=0.6)
plt.axhline(0, color="black", linestyle="--")
plt.xlabel("Actual N")
plt.ylabel("Residual (actual minus predicted)")
plt.title(f"Nitrogen residuals: {SELECTED_MODEL}")
plt.tight_layout()
plt.show()
print("Selected model:", SELECTED_MODEL)

## 8. Refit and predict the competition test set

Refit the selected model on every labeled row. The preprocessing pipeline also refits on those rows, and then transforms the test set with the exact same fitted rules. `Test.csv` supplies features only; the competition does not provide ground-truth test labels locally.

In [ ]:
if SELECTED_MODEL not in MODEL_FACTORIES:
    raise KeyError(f"Choose one of {list(MODEL_FACTORIES)}")
final_model = MODEL_FACTORIES[SELECTED_MODEL]().fit(X, y)
test_predictions = np.asarray(final_model.predict(X_test))
if test_predictions.shape != (len(test_df), len(TARGET_COLUMNS)):
    raise ValueError(f"Unexpected test prediction shape: {test_predictions.shape}")
if not np.isfinite(test_predictions).all():
    raise ValueError("Test predictions contain non-finite values.")

predicted_ppm = pd.DataFrame(test_predictions, columns=TARGET_COLUMNS, index=test_df.index)
predicted_ppm.insert(0, ID_COLUMN, test_df[ID_COLUMN].to_numpy())
predicted_ppm["BulkDensity"] = pd.to_numeric(test_df["BulkDensity"], errors="coerce").to_numpy()
if predicted_ppm["BulkDensity"].isna().any():
    raise ValueError("BulkDensity is required and must be numeric for every test sample.")
display(predicted_ppm.head())

## 9. Build and verify submission

Preserve the original notebook's assumed `20 cm` soil depth and conversion: `kg/ha = ppm × depth_cm × BulkDensity × 0.1`; `Gap = Required − available_kg/ha`. Match predictions to the requested `PID` and `Nutrient` pairs in `Gap_Test.csv` and retain its row order. Check the competition's current rules before changing the physical conversion or clipping predictions.

In [ ]:
long_predictions = predicted_ppm.melt(
    id_vars=[ID_COLUMN, "BulkDensity"],
    value_vars=TARGET_COLUMNS,
    var_name="Nutrient",
    value_name="Available_Nutrients_in_ppm",
)
gap_data = gap_test_df.merge(
    long_predictions, on=[ID_COLUMN, "Nutrient"], how="left", validate="one_to_one", sort=False
)
if len(gap_data) != len(gap_test_df) or gap_data["Available_Nutrients_in_ppm"].isna().any():
    raise ValueError("Some Gap_Test.csv PID/Nutrient pairs have no test prediction.")

required = pd.to_numeric(gap_data["Required"], errors="coerce")
if required.isna().any():
    raise ValueError("Gap_Test.csv Required values must be numeric.")
available_kg_ha = (
    gap_data["Available_Nutrients_in_ppm"]
    * SOIL_DEPTH_CM
    * gap_data["BulkDensity"]
    * 0.1
)
submission = pd.DataFrame({
    "ID": gap_data[ID_COLUMN].astype(str) + "_" + gap_data["Nutrient"].astype(str),
    "Gap": required - available_kg_ha,
})
if submission["ID"].duplicated().any() or not np.isfinite(submission["Gap"]).all():
    raise ValueError("Submission IDs must be unique and every Gap must be finite.")
submission.to_csv(OUTPUT_PATH, index=False)
print(f"Saved {len(submission):,} rows to {OUTPUT_PATH.resolve()}")
display(submission.head())